# 03 — Scoring Evaluation
Score distribution across 89 patient drawings and impairment level breakdown.

In [ ]:
import sys
sys.path.append('..')

import json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path
from src.preprocessing.pipeline import PreprocessingPipeline
from src.algorithm.comparator import ShapeComparator
from src.algorithm.detector import BGTErrorDetector
from src.evaluation.scorer import BGTScorer

In [ ]:
# Run full pipeline on all 89 images
pipeline = PreprocessingPipeline()
comparator = ShapeComparator(template_dir='../data/templates/')
detector = BGTErrorDetector()
scorer = BGTScorer()

all_scores = []
all_levels = []

for img_path in Path('../data/raw_drawings/').glob('*.png'):
    preprocessed = pipeline.run(str(img_path))
    comparison   = comparator.compare(preprocessed)
    errors       = detector.detect(preprocessed, comparison)
    report       = scorer.score(errors)
    all_scores.append(report['total_score'])
    all_levels.append(report['impairment_level'])

print(f'Processed {len(all_scores)} images')
print(f'Mean score: {np.mean(all_scores):.2f}  |  Std: {np.std(all_scores):.2f}')

In [ ]:
# Plot 1: Score distribution histogram
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('BGT Score Distribution — 89 Patient Dataset', fontsize=13)

ax1.hist(all_scores, bins=range(0, 20), color='#4a90d9', edgecolor='white', alpha=0.85)
ax1.set_xlabel('Total Score (0–18)')
ax1.set_ylabel('Number of Patients')
ax1.set_title('Score Histogram')
ax1.axvline(np.mean(all_scores), color='red', linestyle='--', label=f'Mean = {np.mean(all_scores):.1f}')
ax1.legend()

# Plot 2: Impairment level pie chart
level_counts = Counter(all_levels)
labels = [l.replace('_', ' ').title() for l in level_counts.keys()]
colors = ['#2ecc71','#f1c40f','#e67e22','#e74c3c','#8e44ad']
ax2.pie(level_counts.values(), labels=labels, colors=colors[:len(labels)],
        autopct='%1.1f%%', startangle=90)
ax2.set_title('Impairment Level Distribution')

plt.tight_layout()
plt.savefig('../assets/score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()